# PetBreed AI training

Select a GPU runtime when available, then run the cells in order. Sign in to mount Google Drive. Each run saves its checkpoint, metrics, and environment details to a new Drive folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY_URL = 'https://github.com/nikalab7/Petdetect-.git'
PROJECT_DIR = Path('/content/Petdetect-')
if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Device: {DEVICE}; source commit: {COMMIT}')

In [ ]:
import importlib.metadata
import json
import sys
from datetime import datetime, timezone

RUN_DIR = Path('/content/drive/MyDrive/PetBreedAI/runs') / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_utc')
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATA_DIR = Path('/content/PetBreedAI-data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
packages = ['torch', 'torchvision', 'scikit-learn', 'numpy', 'pillow', 'streamlit', 'pytest']
metadata = {
    'source_commit': COMMIT,
    'python_version': sys.version,
    'device': DEVICE,
    'cuda_version': torch.version.cuda,
    'packages': {name: importlib.metadata.version(name) for name in packages},
}
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Results will be saved in: {RUN_DIR}')

In [ ]:
command = [
    sys.executable, str(PROJECT_DIR / 'train.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(RUN_DIR),
    '--head-epochs', '3', '--tune-epochs', '3',
    '--batch-size', '32', '--seed', '42', '--device', 'auto',
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)

results = json.loads((RUN_DIR / 'train_results.json').read_text(encoding='utf-8'))
print(f"Selected stage: {results['selected_stage']}")
print(f"Best validation macro-F1: {results['best_validation_macro_f1']:.4f}")
print(f"Run folder: {RUN_DIR}")